In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import classification_report


In [11]:
import re
import spacy
import nltk
from nltk.corpus import stopwords

# setup
nltk.download('stopwords')
nlp = spacy.load("en_core_web_sm")
nltk_stop_words = set(stopwords.words('english'))

def advanced_nlp_clean(text):
    if not isinstance(text, str): 
        return ""
    
    text = re.sub(r'\{.*?\}', '', text.lower())
    text = re.sub(r'\b\d+\.\d+(\.\d+)?\b', '', text)
    text = re.sub(r'[\n\t\r]', ' ', text)

    doc = nlp(text)
    
    tokens = []
    for token in doc:
        if token.is_alpha:
            if token.text in ["not", "no", "never"]:
                tokens.append(token.text)
            elif token.text not in nltk_stop_words:
                tokens.append(token.lemma_)
    
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ravid\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
#load clean_dataset
df = pd.read_csv("cleaned_ticket.csv")
print(df.shape)
df.head()

(8469, 18)


,ticket id,customer name,customer email,customer age,customer gender,product purchased,date of purchase,category,ticket subject,ticket_text,ticket status,resolution,priority,ticket channel,first response time,time to resolution,customer satisfaction rating,clean_text
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,critical,Social media,2023-06-01 12:15:36,NaN,NaN,issue please assist billing zip code appreciat...
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,critical,Chat,2023-06-01 16:45:38,NaN,NaN,issue please assist need change exist product ...
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,face problem not turn work fine yesterday resp...
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,issue please assist problem interested love se...
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,issue please assist note seller not responsibl...


In [13]:
#feature and labels

X = df["clean_text"]

y_category = df["category"]
y_priority = df["priority"]

In [14]:
#Test and train the daa

X_train, X_test, y_cat_train, y_cat_test = train_test_split(
    X, y_category, test_size=0.2, random_state=42
)

_, _, y_pri_train, y_pri_test = train_test_split(
    X, y_priority, test_size=0.2, random_state=42
)

In [15]:
#category model

category_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ("clf", LogisticRegression(max_iter=200))
])

category_model.fit(X_train, y_cat_train)

,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [16]:
#priority model
priority_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ("clf", LogisticRegression(max_iter=200))
])

priority_model.fit(X_train, y_pri_train)

,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [17]:
#Evaluation
print("CATEGORY REPORT\n")
print(classification_report(y_cat_test, category_model.predict(X_test)))

print("\nPRIORITY REPORT\n")
print(classification_report(y_pri_test, priority_model.predict(X_test)))

CATEGORY REPORT

                      precision    recall  f1-score   support

     billing inquiry       0.17      0.13      0.15       357
cancellation request       0.23      0.24      0.23       327
     product inquiry       0.19      0.21      0.20       316
      refund request       0.20      0.23      0.21       345
     technical issue       0.22      0.23      0.22       349

            accuracy                           0.20      1694
           macro avg       0.20      0.21      0.20      1694
        weighted avg       0.20      0.20      0.20      1694


PRIORITY REPORT

              precision    recall  f1-score   support

    critical       0.21      0.21      0.21       411
        high       0.27      0.30      0.28       409
         low       0.23      0.23      0.23       415
      medium       0.31      0.29      0.30       459

    accuracy                           0.26      1694
   macro avg       0.26      0.26      0.26      1694
weighted avg       0.26 

In [18]:
#pridiction function
def predict_ticket(text):
    cleaned = advanced_nlp_clean(text)

    cat = category_model.predict([cleaned])[0]
    pri = priority_model.predict([cleaned])[0]

    # 🔥 RULE-BASED BOOST (CRITICAL)
    urgent_words = ["angry", "refund", "money back", "not working", "broken", "worst"]

    if any(word in cleaned for word in urgent_words):
        pri = "high"

    if "refund" in cleaned or "money back" in cleaned:
        cat = "complaint"

    return cleaned, cat, pri

In [19]:
#test case
text = "I am so angry! TV not working, want money back"

cleaned, cat, pri = predict_ticket(text)

print("\n--- LIVE TEST ---")
print("Original:", text)
print("Cleaned:", cleaned)
print("Category:", cat)
print("Priority:", pri)


--- LIVE TEST ---
Original: I am so angry! TV not working, want money back
Cleaned: angry tv not work want money back
Category: complaint
Priority: high
